# GAE Training and Evaluation — Google Colab

This is a Colab-ready experimental counterpart of notebook 6. It trains the current reusable `AttributeAwareGAE`, evaluates thresholded anomaly detection, preserves results in Google Drive, and supports controlled additional experiments.

## 1 — Colab Runtime, Repository, and Drive Configuration

Set the controls below before running. `train-only` uses a prepared graph bundle; `full` rebuilds the pipeline from raw logs and requires the optional Azure secrets when LLM enrichment is enabled.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import gzip
import json
import os
import shutil
import subprocess
import sys
import time

IN_COLAB = "google.colab" in sys.modules
REPOSITORY_URL = "https://github.com/michaail/hybrid-logs-analyzer.git"
GIT_REF = "main"

DATASET = "hdfs"  # "hdfs" or "bgl"
RUN_MODE = "train-only"  # "train-only" or "full"
RUN_ID_PREFIX = "hdfs_gae"
INPUT_RUN_ID = None
GRAPH_DATASET_RELATIVE_PATH = (
    "data/processed/hdfs/"
    "20260818_0002_1_parser_3_graph_dataset.pt.gz"
)
REUSE_DRIVE_CACHE = True
RUN_EXPERIMENT_MATRIX = False

REPO_ROOT = Path("/content/hybrid-logs-analyzer") if IN_COLAB else Path.cwd().parents[1]
WORKSPACE_ROOT = Path("/content/workspace") if IN_COLAB else REPO_ROOT
DRIVE_ROOT = Path("/content/drive/MyDrive/hybrid-log-analyzer-artifacts")
print(f"Running in Google Colab: {IN_COLAB}")

## 2 — Clone or Update the Repository

In [ ]:
if IN_COLAB:
    from google.colab import drive, userdata
    drive.mount("/content/drive")
    if not REPO_ROOT.exists():
        subprocess.check_call([
            "git", "clone", "--depth", "1", "--branch", GIT_REF,
            REPOSITORY_URL, str(REPO_ROOT),
        ])
    else:
        subprocess.check_call(["git", "-C", str(REPO_ROOT), "fetch", "--depth", "1", "origin", GIT_REF])
        subprocess.check_call(["git", "-C", str(REPO_ROOT), "checkout", GIT_REF])
        subprocess.check_call(["git", "-C", str(REPO_ROOT), "pull", "--ff-only", "origin", GIT_REF])
    DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

WORKSPACE_ROOT.mkdir(parents=True, exist_ok=True)
os.environ["PIPELINE_WORKSPACE_ROOT"] = str(WORKSPACE_ROOT)
print(f"Code checkout: {REPO_ROOT}")
print(f"Workspace: {WORKSPACE_ROOT}")

## 3 — Install Colab-Compatible Dependencies

In [ ]:
if IN_COLAB:
    subprocess.check_call([
        sys.executable, str(REPO_ROOT / "scripts" / "install_colab.py"),
        "--project-root", str(REPO_ROOT),
    ])
else:
    print("Local execution: use the project's virtual environment and requirements.txt.")

## 4 — Configure Python Paths and Verify GPU Availability

In [ ]:
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)

import torch
print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()} | devices: {torch.cuda.device_count()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Available GPU memory: {torch.cuda.mem_get_info()[0] / 1024**3:.1f} GiB")
elif IN_COLAB:
    raise RuntimeError("No GPU detected. Select Runtime → Change runtime type → GPU, then restart.")

## 5 — Load Optional Azure OpenAI Secrets

Secrets are only needed for `RUN_MODE = "full"` when enrichment is enabled. They are never printed or written to Drive.

In [ ]:
AZURE_SECRET_KEYS = [
    "AZURE_OPENAI_ENDPOINT", "AZURE_OPENAI_API_KEY", "AZURE_OPENAI_API_VERSION",
    "AZURE_OPENAI_DEPLOYMENT_MISTRAL_LARGE", "AZURE_OPENAI_DEPLOYMENT_MISTRAL_SMALL",
]
if IN_COLAB:
    loaded_secret_names = []
    for name in AZURE_SECRET_KEYS:
        value = userdata.get(name)
        if value:
            os.environ[name] = value
            loaded_secret_names.append(name)
    print(f"Loaded {len(loaded_secret_names)} optional Azure secrets.")
else:
    from dotenv import load_dotenv
    load_dotenv(REPO_ROOT / ".env")

## 6 — Stage Raw Data, Processed Graph Data, and Cached Artifacts from Drive

In [ ]:
def stage_from_drive(relative_path: str) -> Path:
    source = DRIVE_ROOT / relative_path
    destination = WORKSPACE_ROOT / relative_path
    if not source.exists():
        raise FileNotFoundError(f"Missing Drive artifact: {source}")
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source, destination)
    return destination


def stage_tree_from_drive(relative_path: str) -> None:
    source = DRIVE_ROOT / relative_path
    destination = WORKSPACE_ROOT / relative_path
    if source.exists():
        destination.mkdir(parents=True, exist_ok=True)
        subprocess.check_call(["rsync", "-a", "--partial", f"{source}/", f"{destination}/"])


def stage_graph_dataset(relative_path: str) -> Path:
    staged = stage_from_drive(relative_path) if IN_COLAB else WORKSPACE_ROOT / relative_path
    if staged.suffix != ".gz":
        return staged
    extracted = staged.with_suffix("")
    if not extracted.exists():
        print(f"Decompressing {staged.name} → {extracted.name}")
        with gzip.open(staged, "rb") as source, open(extracted, "wb") as destination:
            shutil.copyfileobj(source, destination, length=16 * 1024 * 1024)
    return extracted

if IN_COLAB and REUSE_DRIVE_CACHE:
    stage_tree_from_drive(f"artifacts/cache/{DATASET}")
    stage_tree_from_drive("artifacts/runs")

if RUN_MODE == "train-only":
    if INPUT_RUN_ID:
        manifest = json.loads(stage_from_drive(f"artifacts/runs/{INPUT_RUN_ID}.json").read_text())
        GRAPH_DATASET_RELATIVE_PATH = manifest["artifacts"]["graph_dataset"]
    GRAPH_DATASET_PATH = stage_graph_dataset(GRAPH_DATASET_RELATIVE_PATH)
else:
    raw_name = "BGL_full.log" if DATASET == "bgl" else "HDFS_full.log"
    stage_from_drive(f"data/raw/{raw_name}")
    if DATASET == "hdfs":
        stage_from_drive("data/raw/anomaly_label.csv")
    GRAPH_DATASET_PATH = None
print(f"Graph dataset: {GRAPH_DATASET_PATH}")

## 7 — Define GAE Training Experiments

The `--set` entries below override the versioned baseline configuration. The active model exposes hidden/latent dimensions, learning rate, batch size, epochs, clean/noisy training, loss weights, GINE aggregation, and node transformation. Dropout, weight decay, and early stopping are not current `AttributeAwareGAE` options, so they are intentionally not passed.

## 8 — Validate Experimental Controls

Only implemented configuration fields are accepted below. This avoids recording an ablation setting that the reusable pipeline cannot apply.

In [ ]:
BASE_CONFIG_PATH = REPO_ROOT / "configs" / "ablation_base.yaml"
EXPERIMENTS = [
    {"name": "baseline", "overrides": {
        "training.train_mode": "clean", "training.hidden_dim": 128,
        "training.latent_dim": 64, "training.batch_size": 256,
        "training.epochs": 25, "training.learning_rate.hdfs": 0.01,
        "training.alpha": 1.0, "training.beta": 1.0, "training.gamma": 1.0,
        "training.pre_normalize_edges": True,
        "ablation.graph.gine_aggregation": "sum",
        "ablation.graph.node_transformation": "mlp",
    }},
    {"name": "latent_32", "overrides": {"training.latent_dim": 32}},
    {"name": "hidden_256", "overrides": {"training.hidden_dim": 256}},
    {"name": "learning_rate_001", "overrides": {"training.learning_rate.hdfs": 0.001}},
    {"name": "edge_weight_05", "overrides": {"training.gamma": 0.5}},
]
UNAVAILABLE_CONTROLS = {
    "dropout": "not implemented by AttributeAwareGAE",
    "weight_decay": "not implemented by the reusable training optimizer",
    "early_stopping": "the pipeline selects the best validation-F1 epoch but runs all requested epochs",
}
SUPPORTED_EXACT_KEYS = {
    "training.train_mode", "training.hidden_dim", "training.latent_dim",
    "training.batch_size", "training.epochs", "training.alpha", "training.beta",
    "training.gamma", "training.pre_normalize_edges", "training.minimum_edge_std",
    "training.test_run", "training.test_samples", "training.learning_rate.hdfs",
    "training.learning_rate.bgl", "ablation.graph.gine_aggregation",
    "ablation.graph.node_transformation", "ablation.graph.use_edge_features",
    "ablation.embeddings.tfidf_enabled", "ablation.embeddings.sbert_enabled",
    "experiment.seed",
}
active_experiments = EXPERIMENTS if RUN_EXPERIMENT_MATRIX else EXPERIMENTS[:1]
names = [experiment["name"] for experiment in active_experiments]
if DATASET not in {"hdfs", "bgl"}:
    raise ValueError("DATASET must be 'hdfs' or 'bgl'.")
if RUN_MODE not in {"train-only", "full"}:
    raise ValueError("RUN_MODE must be 'train-only' or 'full'.")
if len(names) != len(set(names)):
    raise ValueError("Experiment names must be unique.")
for experiment in active_experiments:
    if not experiment["name"].replace("_", "").replace("-", "").isalnum():
        raise ValueError(f"Invalid experiment name: {experiment['name']}")
    unsupported = set(experiment["overrides"]) - SUPPORTED_EXACT_KEYS
    if unsupported:
        raise ValueError(f"Unsupported overrides in {experiment['name']}: {sorted(unsupported)}")
print(f"Running {len(active_experiments)} experiment(s): {', '.join(names)}")
print("Unavailable controls:", "; ".join(f"{key}: {value}" for key, value in UNAVAILABLE_CONTROLS.items()))

## 9 — Run Training Experiments

Each experiment runs in an isolated process. Completed artifacts are synchronized even if a later experiment fails.

In [ ]:
def sync_to_drive() -> None:
    if not IN_COLAB:
        return
    for directory_name in ("artifacts", "models", "outputs", "runs", "logs"):
        source = WORKSPACE_ROOT / directory_name
        if source.exists():
            destination = DRIVE_ROOT / directory_name
            destination.mkdir(parents=True, exist_ok=True)
            subprocess.run(["rsync", "-a", "--partial", f"{source}/", f"{destination}/"], check=True)


def run_experiment(experiment: dict) -> dict:
    run_id = f"{RUN_ID_PREFIX}_{experiment['name']}"
    command = [
        sys.executable, str(REPO_ROOT / "run_ablation.py"),
        "--mode", RUN_MODE,
        "--config", str(BASE_CONFIG_PATH),
        "--workspace-root", str(WORKSPACE_ROOT),
        "--code-root", str(REPO_ROOT),
        "--checkpoint-root", str(DRIVE_ROOT),
        "--run-id", run_id,
        "--set", f"experiment.dataset={DATASET}",
    ]
    if RUN_MODE == "train-only":
        command.extend(["--graph-dataset", str(GRAPH_DATASET_PATH)])
    elif INPUT_RUN_ID:
        command.extend(["--input-run-id", INPUT_RUN_ID])
    for key, value in experiment["overrides"].items():
        command.extend(["--set", f"{key}={json.dumps(value)}"])

    started = time.monotonic()
    completed = subprocess.run(command, text=True)
    record = {
        "name": experiment["name"], "run_id": run_id,
        "status": "OK" if completed.returncode == 0 else "FAILED",
        "returncode": completed.returncode,
        "elapsed_min": round((time.monotonic() - started) / 60, 2),
        "overrides": experiment["overrides"],
        "metrics_path": str(WORKSPACE_ROOT / "outputs" / DATASET / run_id / "metrics.json"),
    }
    if completed.returncode:
        record["error"] = f"run_ablation.py exited with {completed.returncode}"
    return record

all_results = []
try:
    for experiment in active_experiments:
        result = run_experiment(experiment)
        all_results.append(result)
        print(f"[{result['status']}] {result['name']} ({result['elapsed_min']:.2f} min)")
finally:
    sync_to_drive()

if any(result["status"] == "FAILED" for result in all_results):
    print("One or more experiments failed; successful runs were still checkpointed to Drive.")

## 10 — Save and Verify Durable Experiment Artifacts

The runner checkpoints model, metrics, manifests, and cached stage outputs. This cell can be rerun after an interruption to copy all available local artifacts to Drive.

In [ ]:
sync_to_drive()
for result in all_results:
    metrics_file = Path(result["metrics_path"])
    print(f"{result['run_id']}: metrics {'found' if metrics_file.exists() else 'not found'} at {metrics_file}")
if IN_COLAB:
    print(f"Durable artifact root: {DRIVE_ROOT}")

## 11 — Load an Existing Run for Analysis

Set `ANALYSIS_RUN_IDS` to durable run IDs when analysing an earlier Colab session. Leave it as `None` to analyse the runs completed above.

In [ ]:
ANALYSIS_RUN_IDS = None  # Example: ["hdfs_gae_baseline"]

if ANALYSIS_RUN_IDS:
    for run_id in ANALYSIS_RUN_IDS:
        local_manifest = WORKSPACE_ROOT / "artifacts" / "runs" / f"{run_id}.json"
        if not local_manifest.exists() and IN_COLAB:
            stage_from_drive(f"artifacts/runs/{run_id}.json")
        if not local_manifest.exists():
            raise FileNotFoundError(f"No local or Drive manifest for run {run_id!r}")
    analysis_run_ids = list(ANALYSIS_RUN_IDS)
else:
    analysis_run_ids = [result["run_id"] for result in all_results if result["status"] == "OK"]
if not analysis_run_ids:
    raise RuntimeError("No successful runs are available for analysis.")
print("Analysis runs:", ", ".join(analysis_run_ids))

## 12 — Load and Consolidate Experiment Metrics

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

FIGURE_DIR = WORKSPACE_ROOT / "outputs" / DATASET / "notebook_figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)


def load_manifest(run_id: str) -> dict:
    path = WORKSPACE_ROOT / "artifacts" / "runs" / f"{run_id}.json"
    if not path.exists() and IN_COLAB:
        path = stage_from_drive(f"artifacts/runs/{run_id}.json")
    return json.loads(path.read_text())


def save_figure(figure, name: str) -> Path:
    destination = FIGURE_DIR / name
    figure.savefig(destination, dpi=160, bbox_inches="tight")
    print(f"Saved figure: {destination}")
    return destination

manifests = [load_manifest(run_id) for run_id in analysis_run_ids]
rows = []
for manifest in manifests:
    metrics = manifest["metrics"]
    rows.append({
        "run_id": manifest["run_id"],
        "dataset": manifest["dataset"],
        "mode": manifest["mode"],
        "duration_min": manifest["elapsed_minutes"],
        "threshold": metrics["best_threshold"],
        "val_f1": metrics["val_f1"],
        "val_pr_auc": metrics["val_pr_auc"],
        "val_roc_auc": metrics["val_roc_auc"],
        "test_f1": metrics["test_f1"],
        "test_precision": metrics["test_precision"],
        "test_recall": metrics["test_recall"],
        "test_pr_auc": metrics["test_pr_auc"],
        "test_roc_auc": metrics["test_roc_auc"],
        "n_train": metrics["n_train"], "n_val": metrics["n_val"], "n_test": metrics["n_test"],
    })
comparison = pd.DataFrame(rows).sort_values("test_f1", ascending=False).reset_index(drop=True)
display(comparison)
comparison.to_csv(FIGURE_DIR / "experiment_comparison.csv", index=False)
sync_to_drive()

## 13 — Plot Training Histories

Only persisted facts are plotted: total, structure, node, and edge training losses. The current runner does not persist per-epoch validation metrics or validation loss.

In [ ]:
for manifest in manifests:
    history = manifest.get("history", {})
    if not history or not history.get("total"):
        print(f"No history saved for {manifest['run_id']}")
        continue
    epochs = np.arange(1, len(history["total"]) + 1)
    figure, axes = plt.subplots(1, 3, figsize=(18, 4.5))
    axes[0].plot(epochs, history["total"], color="black", linewidth=2, label="total")
    axes[0].set(title="Total training loss", xlabel="Epoch", ylabel="Loss")
    for name, color in (("structure", "#4C72B0"), ("node", "#55A868"), ("edge", "#C44E52")):
        axes[1].plot(epochs, history[name], linewidth=2, label=name, color=color)
    axes[1].set(title="Reconstruction components", xlabel="Epoch", ylabel="Loss")
    per_component = np.vstack([history["structure"], history["node"], history["edge"]])
    proportions = per_component / np.maximum(per_component.sum(axis=0), 1e-12) * 100
    axes[2].stackplot(epochs, proportions, labels=["structure", "node", "edge"], colors=["#4C72B0", "#55A868", "#C44E52"], alpha=0.8)
    axes[2].set(title="Loss-component proportion", xlabel="Epoch", ylabel="Percent", ylim=(0, 100))
    for axis in axes:
        axis.grid(alpha=0.25)
        axis.legend()
    figure.suptitle(manifest["run_id"])
    figure.tight_layout()
    save_figure(figure, f"{manifest['run_id']}_training_history.png")
    plt.show()
sync_to_drive()

## 14 — Re-evaluate a Saved Model and Decompose Test Scores

This restores an actual saved checkpoint, applies its training-split edge normalization, and recomputes structure, node, and edge reconstruction errors on the held-out test split.

In [ ]:
import torch.nn.functional as F
from torch_geometric.loader import DataLoader
from torch_geometric.utils import scatter
from src.modules.models.gae import AttributeAwareGAE

ANALYSIS_RUN_ID = comparison.iloc[0]["run_id"]  # Change to inspect another successful run.
analysis_manifest = load_manifest(ANALYSIS_RUN_ID)


def local_artifact(relative_path: str) -> Path:
    path = Path(relative_path)
    if not path.is_absolute():
        path = WORKSPACE_ROOT / path
    if not path.exists() and IN_COLAB and not Path(relative_path).is_absolute():
        path = stage_from_drive(relative_path)
    if not path.exists():
        raise FileNotFoundError(f"Artifact unavailable locally or on Drive: {relative_path}")
    return path

checkpoint_path = local_artifact(analysis_manifest["artifacts"]["checkpoint"])
graph_path = local_artifact(analysis_manifest["artifacts"]["graph_dataset"])
checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
bundle = torch.load(graph_path, map_location="cpu", weights_only=False)
training = checkpoint["training"]
test_graphs = [bundle["data_list"][int(index)] for index in bundle["idx_test"]]
if training["test_run"]:
    test_graphs = test_graphs[: int(training["test_samples"])]

edge_mean = checkpoint.get("edge_mean")
edge_std = checkpoint.get("edge_std")
if edge_mean is not None and edge_std is not None:
    mean = torch.tensor(edge_mean, dtype=torch.float32)
    std = torch.tensor(edge_std, dtype=torch.float32)
    for graph in test_graphs:
        if getattr(graph, "edge_attr", None) is not None and graph.edge_attr.numel() > 0:
            graph.edge_attr = (graph.edge_attr.float() - mean) / std

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
analysis_model = AttributeAwareGAE(
    node_dim=checkpoint["node_dim"], edge_dim=checkpoint["edge_dim"],
    hidden_dim=checkpoint["hidden_dim"], latent_dim=checkpoint["latent_dim"],
    gine_aggregation=checkpoint["gine_aggregation"],
    node_transformation=checkpoint["node_transformation"],
).to(device)
analysis_model.load_state_dict(checkpoint["model_state_dict"])
analysis_model.eval()


def component_scores(model, graphs, *, batch_size: int, weights: dict) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    all_structure, all_node, all_edge, all_labels = [], [], [], []
    with torch.no_grad():
        for batch in DataLoader(graphs, batch_size=batch_size):
            batch = batch.to(device)
            z, x_norm, edge_attr = model(batch.x, batch.edge_index, batch.edge_attr)
            graph_count = batch.num_graphs
            structure = torch.zeros(graph_count, device=device)
            if batch.edge_index.size(1):
                logits = model.decode_structure(z, batch.edge_index)
                errors = F.binary_cross_entropy_with_logits(logits, torch.ones_like(logits), reduction="none")
                structure = scatter(errors, batch.batch[batch.edge_index[0]], dim=0, reduce="mean", dim_size=graph_count)
            node_errors = F.mse_loss(model.decode_node_features(z), x_norm, reduction="none").mean(dim=1)
            node = scatter(node_errors, batch.batch, dim=0, reduce="mean", dim_size=graph_count)
            edge = torch.zeros(graph_count, device=device)
            if edge_attr is not None and edge_attr.numel():
                errors = F.mse_loss(model.decode_edge_attributes(z, batch.edge_index), edge_attr, reduction="none").mean(dim=1)
                edge = scatter(errors, batch.batch[batch.edge_index[0]], dim=0, reduce="mean", dim_size=graph_count)
            all_structure.append(torch.nan_to_num(structure).cpu())
            all_node.append(torch.nan_to_num(node).cpu())
            all_edge.append(torch.nan_to_num(edge).cpu())
            all_labels.append(batch.y.cpu())
    return tuple(tensor.numpy() for tensor in (
        torch.cat(all_structure), torch.cat(all_node), torch.cat(all_edge), torch.cat(all_labels)
    ))

weights = {key: float(training[key]) for key in ("alpha", "beta", "gamma")}
test_structure, test_node, test_edge, test_labels = component_scores(
    analysis_model, test_graphs, batch_size=int(training["batch_size"]), weights=weights
)
test_scores = weights["alpha"] * test_structure + weights["beta"] * test_node + weights["gamma"] * test_edge
test_predictions = (test_scores > float(checkpoint["best_threshold"])).astype(int)
print(f"Re-evaluated {len(test_labels):,} held-out test graphs for {ANALYSIS_RUN_ID}.")

## 15 — Test Curves, Score Distributions, and Contribution Analysis

In [ ]:
from sklearn.metrics import (
    ConfusionMatrixDisplay, average_precision_score, classification_report,
    confusion_matrix, precision_recall_curve, roc_auc_score, roc_curve,
)

threshold = float(checkpoint["best_threshold"])
print(classification_report(test_labels, test_predictions, digits=4, zero_division=0))

figure, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for label, color, title in ((0, "#4C72B0", "Normal"), (1, "#C44E52", "Anomaly")):
    subset = test_scores[test_labels == label]
    if len(subset):
        axes[0].hist(subset, density=True, bins=40, alpha=0.45, color=color, label=f"{title} (n={len(subset)})")
axes[0].axvline(threshold, color="black", linestyle="--", label="validation threshold")
axes[0].set(title="Held-out anomaly-score distribution", xlabel="Weighted reconstruction error")
axes[0].legend()
ConfusionMatrixDisplay(confusion_matrix(test_labels, test_predictions, labels=[0, 1]), display_labels=["Normal", "Anomaly"]).plot(ax=axes[1], colorbar=False, cmap="Blues")
axes[1].set_title("Held-out confusion matrix")
figure.tight_layout()
save_figure(figure, f"{ANALYSIS_RUN_ID}_test_distribution_confusion_matrix.png")
plt.show()

if len(np.unique(test_labels)) == 2:
    figure, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    precision, recall, _ = precision_recall_curve(test_labels, test_scores)
    pr_auc = average_precision_score(test_labels, test_scores)
    axes[0].plot(recall, precision, linewidth=2, label=f"PR-AUC = {pr_auc:.4f}")
    axes[0].axhline(test_labels.mean(), color="gray", linestyle="--", label="positive-class rate")
    axes[0].set(xlim=(0, 1), ylim=(0, 1.05), xlabel="Recall", ylabel="Precision", title="Precision–recall curve")
    axes[0].legend()
    false_positive_rate, true_positive_rate, _ = roc_curve(test_labels, test_scores)
    roc_auc = roc_auc_score(test_labels, test_scores)
    axes[1].plot(false_positive_rate, true_positive_rate, linewidth=2, label=f"ROC-AUC = {roc_auc:.4f}")
    axes[1].plot([0, 1], [0, 1], "--", color="gray", label="random")
    axes[1].set(xlim=(0, 1), ylim=(0, 1.05), xlabel="False positive rate", ylabel="True positive rate", title="ROC curve")
    axes[1].legend()
    figure.tight_layout()
    save_figure(figure, f"{ANALYSIS_RUN_ID}_test_pr_roc.png")
    plt.show()
else:
    print("PR/ROC curves require both classes in the selected test split.")

components = pd.DataFrame({
    "Structure": weights["alpha"] * test_structure,
    "Node": weights["beta"] * test_node,
    "Edge": weights["gamma"] * test_edge,
    "label": test_labels,
    "prediction": test_predictions,
})
true_positives = components[(components.label == 1) & (components.prediction == 1)].copy()
if len(true_positives):
    contribution = true_positives[["Structure", "Node", "Edge"]].div(
        true_positives[["Structure", "Node", "Edge"]].sum(axis=1), axis=0
    ).fillna(0)
    average_contribution = contribution.mean().mul(100)
    figure, axes = plt.subplots(1, 2, figsize=(12, 4.5))
    axes[0].pie(average_contribution, labels=average_contribution.index, autopct="%1.1f%%")
    axes[0].set_title(f"Average weighted contribution — true positives (n={len(true_positives)})")
    dominant = contribution.idxmax(axis=1).value_counts().reindex(["Structure", "Node", "Edge"], fill_value=0)
    axes[1].bar(dominant.index, dominant.values, color=["#4C72B0", "#55A868", "#C44E52"])
    axes[1].set(title="Dominant component per true positive", ylabel="Graphs")
    axes[1].grid(axis="y", alpha=0.25)
    figure.tight_layout()
    save_figure(figure, f"{ANALYSIS_RUN_ID}_component_contribution.png")
    plt.show()
else:
    print("No true positives are available for component-contribution analysis.")
components.to_csv(FIGURE_DIR / f"{ANALYSIS_RUN_ID}_test_component_scores.csv", index=False)
sync_to_drive()

## 16 — Split-Leakage Checks and Final Artifact Export

The checks below inspect the saved graph bundle and the restored checkpoint. They verify split identities where available and confirm that clean training retained normal graphs only.

In [ ]:
def split_graphs(index_key: str) -> list:
    return [bundle["data_list"][int(index)] for index in bundle[index_key]]


def graph_ids(graphs: list) -> set[str]:
    identifiers = set()
    for graph in graphs:
        for attribute in ("block_id", "sequence_id", "graph_id"):
            if hasattr(graph, attribute):
                value = getattr(graph, attribute)
                identifiers.add(str(value.item() if hasattr(value, "item") else value))
                break
    return identifiers

raw_train_graphs = split_graphs("idx_train")
val_graphs = split_graphs("idx_val")
original_test_graphs = split_graphs("idx_test")
trained_graphs = raw_train_graphs if training["train_mode"] == "noisy" else [
    graph for graph in raw_train_graphs if int(graph.y.item()) == 0
]
train_ids, val_ids, held_out_ids = map(graph_ids, (raw_train_graphs, val_graphs, original_test_graphs))
print("=== Data leakage verification ===")
if train_ids or val_ids or held_out_ids:
    print(f"Train–validation identifier overlap: {len(train_ids & val_ids)}")
    print(f"Train–test identifier overlap: {len(train_ids & held_out_ids)}")
    print(f"Validation–test identifier overlap: {len(val_ids & held_out_ids)}")
    assert not (train_ids & val_ids or train_ids & held_out_ids or val_ids & held_out_ids), "Split identifiers overlap."
    print("Identifier-disjoint splits: PASS")
else:
    print("No stable graph identifier field is stored; identity-overlap check is unavailable.")

for name, graphs in (("train source", raw_train_graphs), ("train used", trained_graphs), ("validation", val_graphs), ("test", original_test_graphs)):
    labels = np.asarray([int(graph.y.item()) for graph in graphs])
    print(f"{name:12}: {len(labels):,} graphs; anomalies={labels.sum():,}; rate={labels.mean() if len(labels) else 0:.4f}")
if training["train_mode"] == "clean":
    assert all(int(graph.y.item()) == 0 for graph in trained_graphs)
    print("Clean training contains no anomalous graphs: PASS")
else:
    print("Noisy training intentionally retains the original training split.")

batch_norm = analysis_model.raw_node_norm
summary = {
    "analysis_run_id": ANALYSIS_RUN_ID,
    "threshold": threshold,
    "n_recomputed_test": int(len(test_labels)),
    "test_positive_rate": float(np.mean(test_labels)),
    "batch_norm_running_mean_first5": batch_norm.running_mean[:5].detach().cpu().tolist(),
    "batch_norm_running_var_first5": batch_norm.running_var[:5].detach().cpu().tolist(),
}
summary_path = FIGURE_DIR / f"{ANALYSIS_RUN_ID}_analysis_summary.json"
summary_path.write_text(json.dumps(summary, indent=2))
sync_to_drive()
print(f"Analysis summary saved to {summary_path}")